# dmc-navigator-prod on-prem — install smoke test

A runnable copy of the install instructions that pulls the **published ECR image**
(`on-prem/dmc-navigator:stable`) and exercises the whole flow: install → login → pull →
machine-id → license → verify → run.

It runs in a **sandbox `HOME`** (`./.test-home`) so it never touches your real `~/.bashrc` or
`~/.config`. The "issue a license" step is **vendor-only** (needs the private key); customers skip it
and use the JSON Deep-MedChem sends them. Re-run top-to-bottom; the last cell contains opt-in cleanup.

## Installation instructions (production)

What a customer actually does (documented in `README.md`):

```bash
git clone https://github.com/Deep-MedChem/dmc-navigator-on-prem.git
cd dmc-navigator-on-prem
./install_navigator.sh                 # installs the `navigator` command onto your PATH
# open a new shell, then:
AWS_PROFILE=dmc-navigator-source navigator login  # assumes cheese-onprem-pull, then logs in
navigator pull                         # fetch the image
navigator machine-id                   # send the printed integer to Deep-MedChem
navigator update-license license.json  # install the license we send back
navigator verify-license               # -> ✅ Valid license
navigator init --run-dir runs/hk --config-json inputs/HK.json --overwrite
navigator propose --run-dir runs/hk
```

The cells below run exactly this against the published ECR image, in a sandbox `HOME`, and add a
**vendor-only** license-issuing step (which Deep-MedChem does for the customer).

In [ ]:
%pwd

---
## 0. Configuration

In [ ]:
import os, json, pathlib

REPO      = str(pathlib.Path.cwd().resolve())
# Pull source: the published on-prem image in AWS ECR.
IMAGE     = '815935788477.dkr.ecr.us-east-1.amazonaws.com/on-prem/dmc-navigator'
TAG       = 'stable'
AWS_PROFILE = os.environ.get('DMC_NAV_TEST_AWS_PROFILE', 'cheese-admin')
TEST_HOME = os.path.join(REPO, '.test-home')   # sandbox HOME: keeps ~/.bashrc & ~/.config untouched
NAV       = os.path.join(TEST_HOME, '.local/bin/dmc-navigator/navigator')

# VENDOR-ONLY (the "issue a license" step below): the dmc-navigator private key,
# whose public half is baked into the image. Customers never run this step.
KEY_PRIV  = os.path.expanduser('~/.dmc-navigator-keys/dmc-navigator_key-private.pem')
CL_DIR    = os.path.expanduser(os.environ.get(
    'DMC_NAV_LICENSE_TOOL_DIR', '~/Projects/DeepMedChem/cheese-admin/cheese_license'))

os.makedirs(TEST_HOME, exist_ok=True)
assert pathlib.Path(REPO, 'install_navigator.sh').exists(), 'run from the repo checkout'
print('pull source :', f'{IMAGE}:{TAG}')
print('sandbox HOME:', TEST_HOME)
print('AWS profile :', AWS_PROFILE)

## 1. Install the `navigator` command (into the sandbox HOME)

In [ ]:
!HOME={TEST_HOME} bash {REPO}/install_navigator.sh

### Confirm the exact ECR image, then log in and pull
The installed `.env` already contains the shared production ref. `navigator login` uses the active AWS
identity, assumes `arn:aws:iam::815935788477:role/cheese-onprem-pull` when needed, and keeps the
temporary credentials in memory only. The default below is vendor SSO; set
`DMC_NAV_TEST_AWS_PROFILE` before starting Jupyter to exercise a customer source profile.

In [ ]:
envp = pathlib.Path(REPO, '.env')
lines = envp.read_text().splitlines()
def setkv(lines, k, v):
    hit = False
    for i, l in enumerate(lines):
        if l.startswith(k + '='):
            lines[i] = f'{k}={v}'; hit = True
    if not hit: lines.append(f'{k}={v}')
    return lines
lines = setkv(lines, 'DMC_NAV_IMAGE', IMAGE)
lines = setkv(lines, 'DMC_NAV_IMAGE_TAG', TAG)
envp.write_text('\n'.join(lines) + '\n')
print(envp.read_text())

# Let AWS read the real profile/SSO cache, but sandbox Docker's stored ECR token.
# REPO_FOLDER points the sandbox-installed wrapper at this checkout without changing HOME.
!REPO_FOLDER={REPO} DOCKER_CONFIG={TEST_HOME}/.docker AWS_PROFILE={AWS_PROFILE} {NAV} login
!HOME={TEST_HOME} DOCKER_CONFIG={TEST_HOME}/.docker {NAV} pull

## 2. Machine fingerprint
Works before any license exists. Send this integer to Deep-MedChem.

In [ ]:
out = !HOME={TEST_HOME} {NAV} machine-id
MID = [l.strip() for l in out if l.strip().isdigit()][-1]
print('machine-id:', MID)

## 3. Issue a license — *vendor-only (Deep-MedChem does this for the customer)*
In production you run this from `cheese-admin/cheese_license/` and send the resulting JSON to the
customer. Here we sign one for this machine with the vendor private key, inside the image
(which ships pycryptodome), reusing `cheese_license/license_generator.py`. Set
`DMC_NAV_LICENSE_TOOL_DIR` before starting Jupyter if that checkout lives elsewhere. **Requires the
ECR image built
with the matching public key**, so the signature verifies in the next step.

In [ ]:
!docker run --rm --platform linux/amd64 -u $(id -u) -v /:/data --entrypoint python -w /data{CL_DIR} {IMAGE}:{TAG} \
    license_generator.py license -n dmc-navigator -c 'Local Test' -x 2099-01-01 \
    -l {MID} -k /data{KEY_PRIV} > {REPO}/_local_license.json
print(open(f'{REPO}/_local_license.json').read())

## 4. Install & verify the license

In [ ]:
!HOME={TEST_HOME} {NAV} update-license {REPO}/_local_license.json
!HOME={TEST_HOME} {NAV} verify-license

## 5. Run the workflow
Drop a target config in `./inputs`, then init / propose / status. (Here we borrow the example target
shipped inside the image and point it at the in-image example reaction/synthon TSVs.)

In [ ]:
# Borrow examples/configs/HK.json from the image and pin its nested space paths.
!docker run --rm --platform linux/amd64 --entrypoint cat {IMAGE}:{TAG} examples/configs/HK.json > {REPO}/inputs/HK.json
hk = json.load(open(f'{REPO}/inputs/HK.json'))
hk['space']['reactions_path'] = '/opt/dmc-navigator-prod/examples/reactions.tsv'
hk['space']['synthons_path']  = '/opt/dmc-navigator-prod/examples/synthons.tsv'
json.dump(hk, open(f'{REPO}/inputs/HK.json', 'w'), indent=2)

!HOME={TEST_HOME} {NAV} init --run-dir runs/hk --config-json inputs/HK.json --overwrite
!HOME={TEST_HOME} {NAV} propose --run-dir runs/hk
!HOME={TEST_HOME} {NAV} status  --run-dir runs/hk

In [ ]:
import pandas as pd
# propose returns iteration_0000_proposals.parquet and also writes the CSV + score template.
df = pd.read_csv(f'{REPO}/runs/hk/proposals/iteration_0000_proposals.csv')
assert pathlib.Path(REPO, 'runs/hk/proposals/iteration_0000_proposals.parquet').exists()
assert pathlib.Path(REPO, 'runs/hk/scores/iteration_0000_scores_template.csv').exists()
print(f'{len(df)} molecules proposed')
df[['product_id', 'smiles', 'reaction_id']].head()

## 6. Negative checks (fail-closed)
No license, and a license for a different machine, must both be refused.

In [ ]:
# Wrong-machine license -> hardware mismatch
!docker run --rm --platform linux/amd64 -u $(id -u) -v /:/data --entrypoint python -w /data{CL_DIR} {IMAGE}:{TAG} \
    license_generator.py license -n dmc-navigator -c 'Wrong Machine' -x 2099-01-01 \
    -l 12345 -k /data{KEY_PRIV} > {REPO}/_local_license.json
!HOME={TEST_HOME} {NAV} update-license {REPO}/_local_license.json >/dev/null
# NOTE: the `!cmd` capture MUST be on its own line. `rc = !cmd; print(...)` sends
# the `; print(...)` to the shell (bash errors on it), so print never runs.
rc = !HOME={TEST_HOME} {NAV} verify-license
print('\n'.join(rc))

## 7. Optional cleanup
The commands are intentionally commented out so a rerun cannot erase local artifacts by surprise.
Uncomment them after the smoke test. (Real installs: `navigator uninstall`.)

In [ ]:
#!rm -rf {TEST_HOME} {REPO}/_local_license.json {REPO}/license.json {REPO}/runs {REPO}/inputs
#print('cleaned')